### Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import datasets
from sklearn.preprocessing import StandardScaler

# 2. Klasyfikacja

Klasyfikacja to podobnie jak poprzednio poznana regresja przykład **uczenia nadzorowanego**.

Różnica polega na celu: zamiast przewidywać konkretną liczbę (wartość ciągłą), przewidujemy **przynależność obserwacji do jednej z $k$ dyskretnych klas**. W ramach tych zajęć skupimy się na **klasyfikacji binarnej**, czyli uczeniu modeli rozróżniających dwa stany (np. 0 i 1).

## Funkcje kosztu

Zaimplementować funkcje które przyjmują dwa wektory: y_true (faktyczne etykiety) oraz y_pred (przewidywania modelu 0 lub 1) i zwracają metryki.

In [ ]:
def confusion_matrix(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    ...
    return np.array([tp, tn, fp, fn])

def plot_confusion_matrix(y_true, y_pred, class_names=['Negative (0)', 'Positive (1)']):
    tp, tn, fp, fn = confusion_matrix(y_true, y_pred)
    cm_2d = np.array([[tn, fp],
                      [fn, tp]])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_2d, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 16})
    plt.xlabel('Predicted Class', fontsize=14)
    plt.ylabel('True Class', fontsize=14)
    plt.title('Confusion Matrix', fontsize=16)
    plt.tight_layout()
    plt.show()

def accuracy_score(y_true, y_pred):
    ...

def precision_score(y_true, y_pred):
    ...

def recall_score(y_true, y_pred):
    ...

def f1_score(y_true, y_pred):
    p = ...
    r = ...
    if (p + r) == 0:
        return 0.0
    return ...

In [ ]:
y_test_true = [1, 1, 0, 0]
y_test_pred = [1, 0, 1, 0]
cm = confusion_matrix(y_test_true, y_test_pred)
assert len(cm) == 4, "Confusion matrix should have 4 elements"
assert cm[0] == 1, "TP should be index 0"
assert cm[1] == 1, "TN should be index 1"
assert cm[2] == 1, "FP should be index 2"
assert cm[3] == 1, "FN should be index 3"
assert accuracy_score(y_test_true, y_test_pred) == 0.5
assert precision_score(y_test_true, y_test_pred) == 0.5
assert recall_score(y_test_true, y_test_pred) == 0.5
assert f1_score(y_test_true, y_test_pred) == 0.5

## Regresja logistyczna


### Modele Decyzyjne vs. Probabilistyczne

Przyjmując zadanie klasyfikacji binarnej, możemy podzielić algorytmy ze względu na rodzaj zwracanej informacji:

### **Modele probabilistyczne** (np. Regresja Logistyczna)
Modele te nie tylko wskazują klasę, ale zadają nam **rozkład prawdopodobieństwa** na etykietach.

Zamiast samej klasy, model zwraca wartość $P(y=1|x)$. Pozwala to ocenić stopień pewności predykcji.

Dlaczego może nas interesować cały rozkład prawdopodobieństwa zamiast samej, najbardziej prawdopodobnej odpowiedzi (pomyśl o diagnostyce medycznej lub systemach bezpieczeństwa)?




Usupełnić implementację regresji logistycznej.

In [ ]:
class LogisticRegression:
    def __init__(self):
        self.weights = None
        self.bias = None
        self.losses = []

    def _sigmoid(self, x):
        return ...

    def compute_loss(self, y_true, y_pred):
        epsilon = 1e-9
        y1 = ... # calculate the loss for the positive class (where y=1)
        y2 = ... # calculate the loss for the negative class (where y=0)
        return ... # return the negative average of the combined losses

    def feed_forward(self,X):
        z = ... # linear combination of inputs and weights
        A = ... # apply the Sigmoid function to the linear result to get probabilities
        return A

    def fit(self, X, y, lr=0.0001, n_iters=1000):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(n_iters):
            A = self.feed_forward(X)
            self.losses.append(self.compute_loss(y,A))
            dz = A - y
            dw = (1 / n_samples) * np.dot(X.T, dz)
            db = (1 / n_samples) * np.sum(dz)
            self.weights ... # update the weights by moving in the opposite direction of the gradient
            self.bias ... # update the bias by moving in the opposite direction of the gradient.

    def predict(self, X):
        threshold = 0.5
        y_hat = np.dot(X, self.weights) + self.bias
        y_predicted = self._sigmoid(y_hat)
        y_predicted_cls = [1 if i > threshold else 0 for i in y_predicted]
        return np.array(y_predicted_cls)

Przetestuj zaimplmentowany model na przykładowym zbiorze danych. Accuracy powinno wynieść 0.95.

In [ ]:
dataset = datasets.load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name='target')
df = pd.concat([X, y], axis=1)
df.head()

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X = pd.DataFrame(X, columns=dataset.feature_names)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, hue='target', palette='magma', legend=False)
plt.title('Class Distribution (0: Malignant, 1: Benign)')
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.show()

In [ ]:
X, y = dataset.data, dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)
model = ...
... # fit
predictions = ...
accuracy = ...
print(f"Model Accuracy: {accuracy:.2f}")

## SVM

### **Modele decyzyjne** (np. SVM)
Modele te wyznaczają twardą granicę między klasami. Dla zadanego przykładu podają nam po prostu jego przewidzianą etykietę.

Nie informują nas, jak bardzo model jest "pewny" swojej decyzji. Otrzymujemy wynik $y \in \{0, 1\}$.

In [ ]:
class LinearSVM:
    def __init__(self, lambda_param=0.01):
        self.weights = None
        self.bias = None
        self.lambda_param = lambda_param
        self.losses = []

    def compute_loss(self, X, y, y_pred):
        # Hinge Loss: max(0, 1 - y_true * y_pred) + L2 regularization
        reg_term = ...
        hinge_loss = ...
        return reg_term + hinge_loss

    def fit(self, X, y, lr=0.001, n_iters=1000):
        n_samples, n_features = X.shape
        y_transformed = np.where(y <= 0, -1, 1)

        self.weights = np.zeros(n_features)
        self.bias = 0
        for _ in range(n_iters):
            linear_output = np.dot(X, self.weights) + self.bias
            self.losses.append(self.compute_loss(X, y_transformed, linear_output))
            dw = np.zeros(n_features)
            db = 0
            for idx, x_i in enumerate(X):
                condition = y_transformed[idx] * (np.dot(x_i, self.weights) + self.bias) >= 1
                if condition:
                    dw += 2 * self.lambda_param * self.weights
                    db += 0
                else:
                    dw += 2 * self.lambda_param * self.weights - np.dot(x_i, y_transformed[idx])
                    db -= y_transformed[idx]

            # Update parameters
            self.weights ...
            self.bias ...

    def predict(self, X):
        approx = ... # (w*x + b)
        return np.where(approx >= 0, 1, 0)

In [ ]:
X, y = dataset.data, dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)
model = LinearSVM()
... # fit
predictions = ...
accuracy = ...
print(f"Model Accuracy: {accuracy:.2f}")

## Algorytm k-najbliższych sąsiadów (kNN)

Jest to jedna z najprostszych, a zarazem najskuteczniejszych metod klasyfikacji nieparametrycznej.
Algorytm nie buduje jawnej funkcji decyzyjnej (tzw. *lazy learner*). Przechowuje wszystkie dane treningowe w pamięci.

Dla nowej, nieznanej obserwacji, algorytm szuka $k$ najbliższych punktów ze zbioru treningowego (najczęściej na podstawie odległości euklidesowej). Nowy punkt zostaje przypisany do tej klasy, która dominuje wśród jego $k$ sąsiadów.

Mała wartość $k$ (np. 1) sprawia, że model jest bardzo czuły na szum. Duża wartość $k$ sprawia, że granice między klasami stają się gładsze, ale model może ignorować mniejsze, istotne skupiska danych. Ponieważ k-NN bazuje na odległości, **niezbędne jest przeskalowanie danych**. W przeciwnym razie cechy o większych zakresach liczbowych zdominują wynik.


In [ ]:
class KNN:
    def __init__(self, k, X, y):
        self.k = k
        self.X_train = X
        self.y_train = y

    def predict(self, X_test):
        # get predictions for every row in test data
        y_pred = ...
        return np.array(y_pred)

    def _get_single_prediction(self, x_test_row):
        distances = ... # get distances of test_row vs all training rows
        k_idx = np.argsort(distances)[:self.k]
        k_labels = ... # get corresponding y-labels of training data
        return ... # return most common label

    def _get_euclidean_distance(self, x1, x2):
        sum_squared_distance = ... # calculate euclidean distance for a row pair
        return np.sqrt(sum_squared_distance)

In [ ]:
X, y = dataset.data, dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)
model = KNN(3, X_train, y_train)
predictions = ...
accuracy = ...
print(f"Model Accuracy: {accuracy:.2f}")